# 03 – Feature Engineering Badgeuse
## HumanForYou – Attrition ML

## SECTION – IMPORT + LOAD

In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

df_base = pd.read_csv(os.path.join(PROCESSED_DIR, 'cleaned_attrition_base.csv'))
in_time = pd.read_csv(os.path.join(RAW_DIR, 'in_out_time', 'in_time.csv'))
out_time = pd.read_csv(os.path.join(RAW_DIR, 'in_out_time', 'out_time.csv'))

in_time.rename(columns={in_time.columns[0]: 'EmployeeID'}, inplace=True)
out_time.rename(columns={out_time.columns[0]: 'EmployeeID'}, inplace=True)

print(f'df_base: {df_base.shape}')
print(f'in_time: {in_time.shape}')
print(f'out_time: {out_time.shape}')

df_base: (4410, 40)
in_time: (4410, 262)
out_time: (4410, 262)


## SECTION – CONVERSION HEURES EN MINUTES

In [2]:
day_cols = [c for c in in_time.columns if c != 'EmployeeID']
n_days = len(day_cols)
print(f'Nombre de jours: {n_days}')

def to_minutes(df, cols):
    """Convertit toutes les colonnes datetime-string en minutes depuis minuit."""
    series_list = []
    for col in cols:
        parsed = pd.to_datetime(df[col], errors='coerce')
        series_list.append(parsed.dt.hour * 60 + parsed.dt.minute + parsed.dt.second / 60)
    return pd.concat(series_list, axis=1, keys=cols)

in_minutes = to_minutes(in_time, day_cols)
out_minutes = to_minutes(out_time, day_cols)

print(f'in_minutes shape: {in_minutes.shape}')
print(f'Exemple ligne 0, jour 1: in={in_minutes.iloc[0, 1]:.1f} min, out={out_minutes.iloc[0, 1]:.1f} min')

Nombre de jours: 261
in_minutes shape: (4410, 261)
Exemple ligne 0, jour 1: in=583.8 min, out=1016.2 min


## SECTION – CALCUL FEATURES HORAIRES

In [3]:
# Duration = out - in (seulement quand les deux existent et out > in)
duration = out_minutes - in_minutes
duration[duration <= 0] = np.nan

# Masques
valid_mask = in_minutes.notna() & out_minutes.notna() & (duration > 0)

features = pd.DataFrame(index=in_time.index)
features['EmployeeID'] = in_time['EmployeeID']

# Moyennes et ecarts-types
features['in_mean_minutes'] = in_minutes.mean(axis=1)
features['out_mean_minutes'] = out_minutes.mean(axis=1)
features['work_duration_mean_minutes'] = duration.mean(axis=1)
features['in_std_minutes'] = in_minutes.std(axis=1)
features['out_std_minutes'] = out_minutes.std(axis=1)

# Missing rates
features['missing_in_rate'] = in_minutes.isna().sum(axis=1) / n_days
features['missing_out_rate'] = out_minutes.isna().sum(axis=1) / n_days

# Valid days count
features['valid_days_count'] = valid_mask.sum(axis=1)

print(f'Features shape: {features.shape}')
print(features.describe().round(2))

Features shape: (4410, 9)
       EmployeeID  in_mean_minutes  out_mean_minutes  \
count     4410.00          4410.00           4410.00   
mean      2205.50           600.00           1062.05   
std       1273.20             1.09             80.43   
min          1.00           596.25            957.04   
25%       1103.25           599.27            999.85   
50%       2205.50           600.01           1044.50   
75%       3307.75           600.70           1101.66   
max       4410.00           604.67           1264.04   

       work_duration_mean_minutes  in_std_minutes  out_std_minutes  \
count                     4410.00         4410.00          4410.00   
mean                       462.05           16.65            24.51   
std                         80.41            0.75             1.13   
min                        357.03           14.09            20.51   
25%                        400.40           16.15            23.76   
50%                        444.41           16.65

Seuil de retard : arrivée après 10 h 00 (600 min). Seuil de départ anticipé : avant 16 h 00 (960 min).

In [4]:
LATE_THRESHOLD = 600   # 10h00
EARLY_THRESHOLD = 960  # 16h00

# Late arrival rate: jours ou arrivee > 10h / jours valides in
in_valid_count = in_minutes.notna().sum(axis=1)
late_count = (in_minutes > LATE_THRESHOLD).sum(axis=1)
features['late_arrival_rate'] = (late_count / in_valid_count).fillna(0)

# Early leave rate: jours ou depart < 16h / jours valides out
out_valid_count = out_minutes.notna().sum(axis=1)
early_count = (out_minutes < EARLY_THRESHOLD).sum(axis=1)
features['early_leave_rate'] = (early_count / out_valid_count).fillna(0)

print(f'Features finales: {features.shape}')
print(f'Colonnes: {list(features.columns)}')

Features finales: (4410, 11)
Colonnes: ['EmployeeID', 'in_mean_minutes', 'out_mean_minutes', 'work_duration_mean_minutes', 'in_std_minutes', 'out_std_minutes', 'missing_in_rate', 'missing_out_rate', 'valid_days_count', 'late_arrival_rate', 'early_leave_rate']


## SECTION – IMPUTATION EMPLOYES SANS JOURS VALIDES

In [5]:
time_feat_cols = [c for c in features.columns if c != 'EmployeeID']

# Flag employes sans aucun jour valide
no_valid = features['valid_days_count'] == 0
features['time_features_imputed'] = no_valid.astype(int)
print(f'Employes sans jour valide: {no_valid.sum()}')

# Imputation par mediane globale
for col in time_feat_cols:
    if features[col].isna().any():
        med = features[col].median()
        features[col] = features[col].fillna(med)
        print(f'  {col}: {features[col].isna().sum()} NaN restants apres imputation mediane ({med:.2f})')

print(f'\nNaN total features: {features[time_feat_cols].isna().sum().sum()}')

Employes sans jour valide: 0

NaN total features: 0


## SECTION – MERGE AVEC DATASET PRINCIPAL

In [6]:
df_final = df_base.merge(features, on='EmployeeID', how='left')

print(f'df_base: {df_base.shape}')
print(f'features: {features.shape}')
print(f'df_final: {df_final.shape}')
print(f'NaN total: {df_final.isna().sum().sum()}')

assert df_final.shape[0] == 4410, f'Lignes inattendues: {df_final.shape[0]}'

df_base: (4410, 40)
features: (4410, 12)
df_final: (4410, 51)
NaN total: 0


## SECTION – EXPORT + VALIDATION

In [7]:
# Imputer les eventuels NaN restants (merge left)
remaining_nan = df_final.isna().sum()
if remaining_nan.sum() > 0:
    print('NaN restants apres merge:')
    print(remaining_nan[remaining_nan > 0])
    for col in df_final.columns:
        if df_final[col].isna().any():
            df_final[col] = df_final[col].fillna(df_final[col].median())

assert df_final.shape[0] == 4410
assert df_final.isna().sum().sum() == 0, f'NaN restants: {df_final.isna().sum().sum()}'

output_path = os.path.join(PROCESSED_DIR, 'attrition_with_time_features.csv')
df_final.to_csv(output_path, index=False)

print(f'df_final.shape: {df_final.shape}')
print(f'NaN total: {df_final.isna().sum().sum()}')
print(f'Export OK : {output_path}')
print(f'\nNouvelles colonnes horaires:')
new_cols = [c for c in df_final.columns if c not in df_base.columns]
for c in new_cols:
    print(f'  {c}: mean={df_final[c].mean():.2f}, std={df_final[c].std():.2f}')

df_final.shape: (4410, 51)
NaN total: 0
Export OK : ..\data\processed\attrition_with_time_features.csv

Nouvelles colonnes horaires:
  in_mean_minutes: mean=600.00, std=1.09
  out_mean_minutes: mean=1062.05, std=80.43
  work_duration_mean_minutes: mean=462.05, std=80.41
  in_std_minutes: mean=16.65, std=0.75
  out_std_minutes: mean=24.51, std=1.13
  missing_in_rate: mean=0.09, std=0.02
  missing_out_rate: mean=0.09, std=0.02
  valid_days_count: mean=236.27, std=5.50
  late_arrival_rate: mean=0.50, std=0.03
  early_leave_rate: mean=0.07, std=0.14
  time_features_imputed: mean=0.00, std=0.00
